# The Math Behind Neural Networks

**Deep Learning Indaba 2026 — Lagos, Nigeria**

**Session Leaders:** Pelumi Victor, Catherine Essuman & Claire David

---

This hands-on notebook accompanies the tutorial slides. We will build intuition for the core mathematics powering neural networks — from linear regression and gradient descent through to backpropagation — by implementing everything from scratch in NumPy before seeing how frameworks like PyTorch handle it for us.

**Outline:**
1. Linear Regression & Gradient Descent
2. Logistic Regression & Classification
3. Neural Network Building Blocks (Neurons, Activations, Loss Functions)
4. Forward Propagation — from scratch
5. Backpropagation — from scratch
6. Putting It All Together: Training a Neural Network
7. Bonus: The Same Thing in PyTorch

**Prerequisites:** Basic Python, some calculus & linear algebra.

**Reference material:** [Claire David's Intro to ML — Neural Networks](https://clairedavid.github.io/intro_to_ml/nn/nn_0_intro.html)

## 0. Setup

In [ ]:
# Install dependencies (all pre-installed on Colab, but just in case)
!pip install numpy matplotlib scikit-learn torch --quiet

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Linear Regression & Gradient Descent

We start with the simplest learning algorithm: fitting a straight line to data.

**Key idea:** A *hypothesis function* $h(x) = wx + b$ maps inputs to predictions. We find the best $w$ and $b$ by minimizing the **Mean Squared Error (MSE)** cost:

$$J(w, b) = \frac{1}{m} \sum_{i=1}^{m} \left( h(x^{(i)}) - y^{(i)} \right)^2$$

### 1.1 Generate Some Data

In [ ]:
# True relationship: y = 3x + 7 + noise
m = 100  # number of samples
X = 2 * np.random.rand(m)
y = 3 * X + 7 + np.random.randn(m) * 0.5

plt.scatter(X, y, s=15, alpha=0.7)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Our dataset (can you guess the line?)')
plt.show()

### 1.2 Cost Function

The MSE tells us how far off our predictions are, on average.

In [ ]:
def mse_cost(X, y, w, b):
    """Compute Mean Squared Error cost."""
    m = len(y)
    predictions = w * X + b
    cost = (1 / m) * np.sum((predictions - y) ** 2)
    return cost

# Test with a bad guess
print(f"Cost with w=0, b=0: {mse_cost(X, y, 0, 0):.2f}")
print(f"Cost with w=3, b=7: {mse_cost(X, y, 3, 7):.2f}  (close to true values!)")

### 1.3 Visualizing the Cost Landscape

Let's see how the cost varies with $w$ and $b$. The minimum of this surface is where we want to end up.

In [ ]:
w_range = np.linspace(-2, 8, 100)
b_range = np.linspace(-2, 14, 100)
W_grid, B_grid = np.meshgrid(w_range, b_range)
Cost_grid = np.array([[mse_cost(X, y, w, b) for w, b in zip(w_row, b_row)]
                       for w_row, b_row in zip(W_grid, B_grid)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour plot
cs = axes[0].contourf(W_grid, B_grid, Cost_grid, levels=30, cmap='viridis')
axes[0].set_xlabel('w')
axes[0].set_ylabel('b')
axes[0].set_title('Cost landscape (contour)')
fig.colorbar(cs, ax=axes[0])

# 3D surface
ax3d = fig.add_subplot(122, projection='3d')
axes[1].remove()
ax3d.plot_surface(W_grid, B_grid, Cost_grid, cmap='viridis', alpha=0.8)
ax3d.set_xlabel('w')
ax3d.set_ylabel('b')
ax3d.set_zlabel('Cost')
ax3d.set_title('Cost landscape (3D)')

plt.tight_layout()
plt.show()

### 1.4 Gradient Descent

We navigate the cost landscape by following the negative gradient — like walking downhill through fog.

**Algorithm:**
1. Start with random $w$, $b$
2. Compute the partial derivatives: $\frac{\partial J}{\partial w}$, $\frac{\partial J}{\partial b}$
3. Update: $w \leftarrow w - \alpha \frac{\partial J}{\partial w}$, $\;b \leftarrow b - \alpha \frac{\partial J}{\partial b}$
4. Repeat until convergence

The **learning rate** $\alpha$ controls step size.

In [ ]:
def gradient_descent_linear(X, y, lr=0.1, n_iters=100):
    """Gradient descent for 1D linear regression."""
    m = len(y)
    w, b = 0.0, 0.0  # initialize
    history = {'w': [w], 'b': [b], 'cost': [mse_cost(X, y, w, b)]}

    for i in range(n_iters):
        # Predictions
        y_pred = w * X + b

        # Gradients
        dw = (2 / m) * np.sum((y_pred - y) * X)
        db = (2 / m) * np.sum(y_pred - y)

        # Update
        w -= lr * dw
        b -= lr * db

        history['w'].append(w)
        history['b'].append(b)
        history['cost'].append(mse_cost(X, y, w, b))

    return w, b, history

w_opt, b_opt, hist = gradient_descent_linear(X, y, lr=0.1, n_iters=200)
print(f"Optimized: w = {w_opt:.4f}, b = {b_opt:.4f}")
print(f"True:      w = 3,        b = 7")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost over iterations
axes[0].plot(hist['cost'])
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost (MSE)')
axes[0].set_title('Cost convergence')

# Fitted line
axes[1].scatter(X, y, s=15, alpha=0.7, label='Data')
x_line = np.linspace(0, 2, 100)
axes[1].plot(x_line, w_opt * x_line + b_opt, 'r-', linewidth=2, label=f'Fit: y = {w_opt:.2f}x + {b_opt:.2f}')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('Fitted line')
axes[1].legend()

plt.tight_layout()
plt.show()

### 1.5 Gradient Descent Trajectory on the Cost Landscape

In [ ]:
plt.contourf(W_grid, B_grid, Cost_grid, levels=30, cmap='viridis', alpha=0.8)
plt.colorbar(label='Cost')
plt.plot(hist['w'], hist['b'], 'r.-', markersize=3, linewidth=1, alpha=0.7)
plt.plot(hist['w'][0], hist['b'][0], 'wo', markersize=10, label='Start')
plt.plot(hist['w'][-1], hist['b'][-1], 'r*', markersize=15, label='End')
plt.xlabel('w')
plt.ylabel('b')
plt.title('Gradient descent trajectory')
plt.legend()
plt.show()

### Exercise 1: Learning Rate Sensitivity

Try different learning rates and observe the effect on convergence.

- `lr = 0.001` (too small?)
- `lr = 0.1` (just right?)
- `lr = 0.5` (too large?)
- `lr = 1.0` (diverges?)

In [ ]:
# YOUR CODE HERE: Try different learning rates and plot their convergence curves
fig, ax = plt.subplots(figsize=(10, 5))

for lr in [0.001, 0.01, 0.1, 0.5]:
    _, _, h = gradient_descent_linear(X, y, lr=lr, n_iters=200)
    ax.plot(h['cost'], label=f'lr = {lr}')

ax.set_xlabel('Iteration')
ax.set_ylabel('Cost')
ax.set_title('Effect of learning rate on convergence')
ax.legend()
ax.set_ylim(0, 80)
plt.show()

---
## 2. Logistic Regression & Classification

For classification, we need outputs between 0 and 1 — enter the **sigmoid function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

And the **Binary Cross-Entropy** loss:

$$\mathcal{L}(\hat{y}, y) = -\left[ y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right]$$

### 2.1 The Sigmoid Function

In [ ]:
def sigmoid(z):
    """Sigmoid activation function."""
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z), linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Decision boundary (0.5)')
plt.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('z')
plt.ylabel('$\sigma(z)$')
plt.title('Sigmoid Function')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 2.2 Logistic Regression from Scratch

In [ ]:
# Generate 2-class dataset
from sklearn.datasets import make_moons

X_cls, y_cls = make_moons(n_samples=300, noise=0.25, random_state=42)

plt.scatter(X_cls[y_cls == 0, 0], X_cls[y_cls == 0, 1], label='Class 0', alpha=0.7, s=20)
plt.scatter(X_cls[y_cls == 1, 0], X_cls[y_cls == 1, 1], label='Class 1', alpha=0.7, s=20)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Classification dataset (two moons)')
plt.legend()
plt.show()

In [ ]:
def logistic_regression(X, y, lr=0.1, n_iters=1000):
    """Train logistic regression with gradient descent."""
    m, n = X.shape
    w = np.zeros(n)
    b = 0.0
    costs = []

    for i in range(n_iters):
        # Forward pass
        z = X @ w + b
        y_pred = sigmoid(z)

        # Binary cross-entropy cost
        eps = 1e-8  # avoid log(0)
        cost = -(1 / m) * np.sum(y * np.log(y_pred + eps) + (1 - y) * np.log(1 - y_pred + eps))
        costs.append(cost)

        # Gradients
        dw = (1 / m) * X.T @ (y_pred - y)
        db = (1 / m) * np.sum(y_pred - y)

        # Update
        w -= lr * dw
        b -= lr * db

    return w, b, costs

w_lr, b_lr, costs_lr = logistic_regression(X_cls, y_cls, lr=1.0, n_iters=2000)

# Accuracy
preds = (sigmoid(X_cls @ w_lr + b_lr) >= 0.5).astype(int)
accuracy = np.mean(preds == y_cls)
print(f"Accuracy: {accuracy:.1%}")

plt.plot(costs_lr)
plt.xlabel('Iteration')
plt.ylabel('Cross-Entropy Cost')
plt.title('Logistic Regression Training')
plt.show()

Notice the accuracy is limited — logistic regression can only draw a **linear decision boundary**. The two-moons dataset is *not* linearly separable. We need something more powerful...

---
## 3. Neural Network Building Blocks

A neural network chains together many "neurons" — each one computes:

$$\hat{y} = f\left( \sum_{j=1}^{n} w_j x_j + b \right)$$

where $f$ is a non-linear **activation function**.

### 3.1 The Perceptron

The simplest neural network: a single neuron with a step function.

**Exercise 2:** Can a single perceptron solve the AND, OR, and XOR logic gates?

In [ ]:
def heaviside(z):
    """Heaviside step function."""
    return (z >= 0).astype(float)

def perceptron(x, w, b):
    """Single perceptron with step activation."""
    return heaviside(np.dot(x, w) + b)

# Truth table inputs
inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

# AND gate: w1=0.5, w2=0.5, b=-1 (from slide 43)
w_and = np.array([0.5, 0.5])
b_and = -1.0
print("AND gate:")
for x in inputs:
    print(f"  {x} -> {perceptron(x, w_and, b_and):.0f}")

# OR gate: w1=1, w2=1, b=-0.5 (from slide 96)
w_or = np.array([1.0, 1.0])
b_or = -0.5
print("\nOR gate:")
for x in inputs:
    print(f"  {x} -> {perceptron(x, w_or, b_or):.0f}")

In [ ]:
# EXERCISE: Try to find w1, w2, b for XOR. Can you do it with a single perceptron?
# XOR truth table: (0,0)->0, (0,1)->1, (1,0)->1, (1,1)->0

# Try your values here:
w_xor = np.array([1.0, 1.0])   # change these!
b_xor = -0.5                    # change this!

print("XOR attempt:")
xor_targets = [0, 1, 1, 0]
for x, target in zip(inputs, xor_targets):
    pred = perceptron(x, w_xor, b_xor)
    match = 'OK' if pred == target else 'WRONG'
    print(f"  {x} -> {pred:.0f}  (expected {target}) {match}")

print("\nSpoiler: A single perceptron CANNOT solve XOR!")
print("XOR is not linearly separable. We need a NETWORK of neurons.")

### 3.2 Activation Functions

Activation functions introduce **non-linearity** — without them, stacking layers would collapse into a single linear transformation.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def tanh_act(z):
    return np.tanh(z)

def leaky_relu(z, alpha=0.01):
    return np.where(z >= 0, z, alpha * z)

z = np.linspace(-5, 5, 300)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

activations = [
    ('Sigmoid', sigmoid(z)),
    ('Tanh', tanh_act(z)),
    ('ReLU', relu(z)),
    ('Leaky ReLU', leaky_relu(z)),
]

for ax, (name, vals) in zip(axes.flat, activations):
    ax.plot(z, vals, linewidth=2)
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(name, fontsize=13)
    ax.grid(True, alpha=0.3)

plt.suptitle('Common Activation Functions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 3.3 Loss Functions

The **loss function** $\mathcal{L}$ measures error for one sample. The **cost function** $J$ averages over all samples.

| Task | Loss | Formula |
|:--|:--|:--|
| Regression | MSE | $(\hat{y} - y)^2$ |
| Binary classification | Binary Cross-Entropy | $-[y\log\hat{y} + (1-y)\log(1-\hat{y})]$ |
| Multi-class | Categorical Cross-Entropy | $-\sum_k y_k \log \hat{y}_k$ |

In [ ]:
# Visualize binary cross-entropy
y_hat = np.linspace(0.001, 0.999, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# When true label y = 1
loss_y1 = -np.log(y_hat)
axes[0].plot(y_hat, loss_y1, 'b-', linewidth=2)
axes[0].set_xlabel('$\hat{y}$ (prediction)')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss when y = 1:  $-\log(\hat{y})$')
axes[0].grid(True, alpha=0.3)

# When true label y = 0
loss_y0 = -np.log(1 - y_hat)
axes[1].plot(y_hat, loss_y0, 'r-', linewidth=2)
axes[1].set_xlabel('$\hat{y}$ (prediction)')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss when y = 0:  $-\log(1-\hat{y})$')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Binary Cross-Entropy Loss', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Forward Propagation — From Scratch

Now we build a **fully connected feedforward neural network** and implement the forward pass.

**General equation for layer $\ell$:**

$$\mathbf{a}^{(i, \ell)} = f\left( (\mathbf{W}^{(\ell)})^T \cdot \mathbf{a}^{(i, \ell-1)} + \mathbf{b}^{(\ell)} \right)$$

where the input features are layer 0: $\mathbf{a}^{(i, 0)} = \mathbf{x}^{(i)}$

### 4.1 Network Architecture

We'll build a network for the two-moons problem:
- **Input layer**: 2 features ($x_1$, $x_2$)
- **Hidden layer 1**: 16 neurons (ReLU)
- **Hidden layer 2**: 8 neurons (ReLU)
- **Output layer**: 1 neuron (Sigmoid)

In [ ]:
def initialize_parameters(layer_dims):
    """
    Initialize weights and biases for a neural network.

    Uses He initialization for weights: W ~ N(0, sqrt(2/n_prev))
    This helps avoid vanishing/exploding gradients with ReLU.

    Args:
        layer_dims: list of layer sizes, e.g. [2, 16, 8, 1]
    Returns:
        params: dict with W1, b1, W2, b2, ...
    """
    params = {}
    L = len(layer_dims)

    for l in range(1, L):
        params[f'W{l}'] = np.random.randn(layer_dims[l-1], layer_dims[l]) * np.sqrt(2.0 / layer_dims[l-1])
        params[f'b{l}'] = np.zeros((1, layer_dims[l]))

    return params

# Our network: 2 -> 16 -> 8 -> 1
layer_dims = [2, 16, 8, 1]
params = initialize_parameters(layer_dims)

print("Network architecture:")
for l in range(1, len(layer_dims)):
    print(f"  Layer {l}: W{l} shape = {params[f'W{l}'].shape}, b{l} shape = {params[f'b{l}'].shape}")

In [ ]:
def relu_forward(z):
    return np.maximum(0, z)

def sigmoid_forward(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def forward_propagation(X, params, layer_dims):
    """
    Full forward pass through the network.

    Hidden layers use ReLU, output layer uses Sigmoid.

    Returns:
        A_last: predictions (m, 1)
        cache: dict of all Z and A values (needed for backprop)
    """
    cache = {'A0': X}  # input = layer 0 activations
    A = X
    L = len(layer_dims) - 1  # number of layers (excluding input)

    for l in range(1, L + 1):
        Z = A @ params[f'W{l}'] + params[f'b{l}']  # z = W^T * a + b
        cache[f'Z{l}'] = Z

        if l < L:
            A = relu_forward(Z)     # hidden layers: ReLU
        else:
            A = sigmoid_forward(Z)  # output layer: Sigmoid

        cache[f'A{l}'] = A

    return A, cache

# Test forward pass
A_out, cache = forward_propagation(X_cls, params, layer_dims)
print(f"Output shape: {A_out.shape}")
print(f"Sample predictions (first 5): {A_out[:5, 0]}")
print(f"\nCached layers: {list(cache.keys())}")

---
## 5. Backpropagation — From Scratch

Backpropagation computes **how much each weight contributed to the error** using the chain rule.

**Key equations:**

Last layer error:
$$\delta^{(L)} = f'(z^{(L)}) \odot \mathcal{L}'(a^{(L)}, y)$$

Recursive error for earlier layers:
$$\delta^{(\ell)} = f'(z^{(\ell)}) \odot \left[ W^{(\ell+1)} \cdot \delta^{(\ell+1)} \right]$$

Gradients:
$$\frac{\partial J}{\partial W^{(\ell)}} = \frac{1}{m} (a^{(\ell-1)})^T \cdot \delta^{(\ell)}$$
$$\frac{\partial J}{\partial b^{(\ell)}} = \frac{1}{m} \sum_i \delta^{(\ell)}$$

### 5.1 Chain Rule Refresher

For nested functions $k(x) = h(f(g(x)))$:

$$k'(x) = h'(f(g(x))) \cdot f'(g(x)) \cdot g'(x)$$

Each layer in a neural network is a function composition — backprop applies the chain rule layer by layer, from output back to input.

In [ ]:
def compute_cost(A_last, y):
    """Binary cross-entropy cost."""
    m = y.shape[0]
    eps = 1e-8
    cost = -(1 / m) * np.sum(
        y * np.log(A_last + eps) + (1 - y) * np.log(1 - A_last + eps)
    )
    return cost

In [ ]:
def relu_backward(dA, Z):
    """Derivative of ReLU: 1 if z > 0, else 0."""
    return dA * (Z > 0).astype(float)

def backward_propagation(y, params, cache, layer_dims):
    """
    Full backward pass through the network.

    Returns:
        grads: dict of dW1, db1, dW2, db2, ...
    """
    grads = {}
    L = len(layer_dims) - 1
    m = y.shape[0]
    y = y.reshape(-1, 1)

    # --- Last layer (sigmoid + BCE) ---
    # For sigmoid + binary cross-entropy, the combined derivative simplifies to:
    # dZ_L = A_L - y
    A_L = cache[f'A{L}']
    dZ = A_L - y  # delta for last layer

    grads[f'dW{L}'] = (1 / m) * cache[f'A{L-1}'].T @ dZ
    grads[f'db{L}'] = (1 / m) * np.sum(dZ, axis=0, keepdims=True)

    # --- Hidden layers (ReLU) ---
    for l in range(L - 1, 0, -1):
        # Propagate error backward: delta^(l) = (delta^(l+1) @ W^(l+1)^T) * f'(z^(l))
        dA = dZ @ params[f'W{l+1}'].T
        dZ = relu_backward(dA, cache[f'Z{l}'])  # element-wise with ReLU derivative

        grads[f'dW{l}'] = (1 / m) * cache[f'A{l-1}'].T @ dZ
        grads[f'db{l}'] = (1 / m) * np.sum(dZ, axis=0, keepdims=True)

    return grads

# Test backprop
grads = backward_propagation(y_cls, params, cache, layer_dims)
print("Gradient shapes:")
for key, val in grads.items():
    print(f"  {key}: {val.shape}")

---
## 6. Putting It All Together: Training Loop

The training loop repeats three steps:
1. **Forward propagation** — compute predictions
2. **Backpropagation** — compute gradients
3. **Gradient descent** — update parameters

In [ ]:
def update_parameters(params, grads, lr, layer_dims):
    """Gradient descent parameter update."""
    L = len(layer_dims) - 1
    for l in range(1, L + 1):
        params[f'W{l}'] -= lr * grads[f'dW{l}']
        params[f'b{l}'] -= lr * grads[f'db{l}']
    return params

In [ ]:
def train_neural_network(X, y, layer_dims, lr=0.1, n_iters=5000, print_every=500):
    """Train a neural network from scratch."""
    params = initialize_parameters(layer_dims)
    costs = []

    for i in range(n_iters):
        # Step 1: Forward propagation
        A_last, cache = forward_propagation(X, params, layer_dims)

        # Compute cost
        cost = compute_cost(A_last, y.reshape(-1, 1))
        costs.append(cost)

        # Step 2: Backpropagation
        grads = backward_propagation(y, params, cache, layer_dims)

        # Step 3: Gradient descent
        params = update_parameters(params, grads, lr, layer_dims)

        if (i + 1) % print_every == 0:
            preds = (A_last >= 0.5).astype(int).flatten()
            acc = np.mean(preds == y)
            print(f"Iter {i+1:5d} | Cost: {cost:.4f} | Accuracy: {acc:.1%}")

    return params, costs

# Train!
layer_dims = [2, 16, 8, 1]
trained_params, training_costs = train_neural_network(
    X_cls, y_cls, layer_dims, lr=0.5, n_iters=5000, print_every=500
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost curve
axes[0].plot(training_costs)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost')
axes[0].set_title('Training loss')

# Decision boundary
h = 0.02
x_min, x_max = X_cls[:, 0].min() - 0.5, X_cls[:, 0].max() + 0.5
y_min, y_max = X_cls[:, 1].min() - 0.5, X_cls[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid_input = np.c_[xx.ravel(), yy.ravel()]

Z_grid, _ = forward_propagation(grid_input, trained_params, layer_dims)
Z_grid = Z_grid.reshape(xx.shape)

axes[1].contourf(xx, yy, Z_grid, levels=[0, 0.5, 1], alpha=0.3, colors=['#4444ff', '#ff4444'])
axes[1].contour(xx, yy, Z_grid, levels=[0.5], colors='black', linewidths=2)
axes[1].scatter(X_cls[y_cls == 0, 0], X_cls[y_cls == 0, 1], label='Class 0', s=20, alpha=0.7)
axes[1].scatter(X_cls[y_cls == 1, 0], X_cls[y_cls == 1, 1], label='Class 1', s=20, alpha=0.7)
axes[1].set_title('Learned decision boundary')
axes[1].legend()

plt.tight_layout()
plt.show()

# Final accuracy
final_preds, _ = forward_propagation(X_cls, trained_params, layer_dims)
final_acc = np.mean((final_preds.flatten() >= 0.5).astype(int) == y_cls)
print(f"\nFinal accuracy: {final_acc:.1%}")
print("Compare this to logistic regression — the neural network learns a non-linear boundary!")

### Exercise 3: Experiment with the Architecture

Try changing the network architecture and observe the effect on the decision boundary:
- `[2, 4, 1]` — smaller network
- `[2, 32, 32, 1]` — deeper network
- `[2, 64, 32, 16, 1]` — even deeper

Also try the [TensorFlow Playground](https://playground.tensorflow.org) to visualize interactively!

In [ ]:
# YOUR CODE HERE: Try different architectures
# Example:
# my_dims = [2, 4, 1]
# my_params, my_costs = train_neural_network(X_cls, y_cls, my_dims, lr=0.5, n_iters=5000, print_every=1000)

---
## 7. Bonus: The Same Thing in PyTorch

Now that you understand every step, let's see how PyTorch automates it with **autograd** (automatic differentiation).

In [ ]:
# Install PyTorch (Colab has it pre-installed)
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    print(f"PyTorch version: {torch.__version__}")
except ImportError:
    !pip install torch --quiet
    import torch
    import torch.nn as nn
    import torch.optim as optim
    print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Convert data to PyTorch tensors
X_tensor = torch.FloatTensor(X_cls)
y_tensor = torch.FloatTensor(y_cls).reshape(-1, 1)

# Define the same architecture: 2 -> 16 -> 8 -> 1
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# Loss and optimizer
criterion = nn.BCELoss()                        # Binary Cross-Entropy
optimizer = optim.SGD(model.parameters(), lr=0.5)  # Stochastic Gradient Descent

# Training loop
pytorch_costs = []
n_epochs = 5000

for epoch in range(n_epochs):
    # Forward pass
    y_pred = model(X_tensor)
    loss = criterion(y_pred, y_tensor)
    pytorch_costs.append(loss.item())

    # Backward pass + update (PyTorch does backprop automatically!)
    optimizer.zero_grad()  # clear previous gradients
    loss.backward()        # compute gradients via autograd
    optimizer.step()       # update weights

    if (epoch + 1) % 1000 == 0:
        with torch.no_grad():
            acc = ((y_pred >= 0.5).float() == y_tensor).float().mean()
        print(f"Epoch {epoch+1:5d} | Loss: {loss.item():.4f} | Accuracy: {acc:.1%}")

# Plot
plt.plot(pytorch_costs)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('PyTorch training loss')
plt.show()

In [ ]:
# Decision boundary from PyTorch model
with torch.no_grad():
    grid_tensor = torch.FloatTensor(grid_input)
    Z_pytorch = model(grid_tensor).numpy().reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NumPy model
axes[0].contourf(xx, yy, Z_grid, levels=[0, 0.5, 1], alpha=0.3, colors=['#4444ff', '#ff4444'])
axes[0].contour(xx, yy, Z_grid, levels=[0.5], colors='black', linewidths=2)
axes[0].scatter(X_cls[y_cls == 0, 0], X_cls[y_cls == 0, 1], s=20, alpha=0.7)
axes[0].scatter(X_cls[y_cls == 1, 0], X_cls[y_cls == 1, 1], s=20, alpha=0.7)
axes[0].set_title('Our NumPy implementation')

# PyTorch model
axes[1].contourf(xx, yy, Z_pytorch, levels=[0, 0.5, 1], alpha=0.3, colors=['#4444ff', '#ff4444'])
axes[1].contour(xx, yy, Z_pytorch, levels=[0.5], colors='black', linewidths=2)
axes[1].scatter(X_cls[y_cls == 0, 0], X_cls[y_cls == 0, 1], s=20, alpha=0.7)
axes[1].scatter(X_cls[y_cls == 1, 0], X_cls[y_cls == 1, 1], s=20, alpha=0.7)
axes[1].set_title('PyTorch implementation')

plt.suptitle('Same math, same result!', fontsize=14)
plt.tight_layout()
plt.show()

print("Both models learn a non-linear decision boundary.")
print("PyTorch automates the backpropagation we coded by hand!")

---
## Summary

In this tutorial, you have:

1. **Implemented linear regression** with gradient descent from scratch
2. **Built logistic regression** and seen its limitations on non-linear data
3. **Explored neural network building blocks**: neurons, activation functions, loss functions
4. **Coded forward propagation** — computing predictions layer by layer
5. **Coded backpropagation** — computing gradients using the chain rule
6. **Trained a neural network** that learns non-linear decision boundaries
7. **Seen how PyTorch** automates all of this with autograd

You now know the math behind neural networks!

### Further resources
- [Claire David's Intro to ML course](https://clairedavid.github.io/intro_to_ml/nn/nn_0_intro.html)
- [TensorFlow Playground](https://playground.tensorflow.org) — interactive visualization
- [3Blue1Brown — Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi) — visual explanations